In [42]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [43]:
BASE = Path.cwd().parent.parent / "ansys_data"
print(BASE)
OUTPUT_DIR = Path.cwd().parent.parent / "data" / "raw"

NORMAL_FILES = {
    "pressure_A": BASE / "normal_pressure_node_a.out",
    "pressure_B": BASE / "normal_pressure_node_b.out",
    "pressure_C": BASE / "pressure-node-c-rfile.out",
    "velocity_A": BASE / "normal_velocity_node_a.out",
    "velocity_B": BASE / "velocity-node-b-rfile.out",
    "velocity_C": BASE / "velocity-node-c-rfile.out",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

/home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/ansys_data


### ANSYS DATA LOADING

In [44]:
def parse_fluent_out(filepath):
    rows = []
    with open(filepath, "r") as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) == 3:
                try:
                    rows.append((int(parts[0]), float(parts[1]), float(parts[2])))
                except ValueError:
                    pass
    return rows


def build_cfd_dataframe(files: dict, run_id: int = 0) -> pd.DataFrame:
    """
    Parse all 6 .out files and merge into one DataFrame.
    Keeps ALL 701 rows — transient + steady-state.
    """
    parsed   = {}
    time_map = {}

    for col, path in files.items():
        rows = parse_fluent_out(path)
        parsed[col]  = {r[0]: r[1] for r in rows}
        time_map     = {r[0]: r[2] for r in rows}

    master_steps = sorted(parsed["pressure_A"].keys())

    records = []
    for step in master_steps:
        records.append({
            "run_id":     run_id,
            "timestep":   step,
            "time_s":     round(time_map[step], 1),
            "pressure_A": parsed["pressure_A"][step],
            "pressure_B": parsed["pressure_B"][step],
            "pressure_C": parsed["pressure_C"][step],
            "velocity_A": parsed["velocity_A"][step],
            "velocity_B": parsed["velocity_B"][step],
            "velocity_C": parsed["velocity_C"][step],
            "label":      0,
            "scenario":   "normal",
        })

    return pd.DataFrame(records)


# Load the single real CFD run
cfd_df = build_cfd_dataframe(NORMAL_FILES, run_id=0)
print(f"Real CFD run loaded: {len(cfd_df)} rows  "
      f"(t={cfd_df.time_s.min()}s → {cfd_df.time_s.max()}s)")
print(cfd_df[["time_s","pressure_A","pressure_B","pressure_C",
              "velocity_A","velocity_B","velocity_C"]].head(5).to_string(index=False))

Real CFD run loaded: 701 rows  (t=0.0s → 70.0s)
 time_s    pressure_A   pressure_B  pressure_C  velocity_A  velocity_B  velocity_C
    0.0      0.007812     0.007812    0.007812    2.951487    2.963145    2.949855
    0.1 117246.900000 58469.720000  359.566900    2.947447    2.952487    2.940701
    0.2 101416.800000 50775.800000  300.451400    2.948791    2.953200    2.941500
    0.3  88645.910000 44387.200000  253.575200    2.949434    2.953305    2.941705
    0.4  82063.770000 41106.620000  224.899000    2.949710    2.953260    2.941774


In [45]:
cfd_df.head(10)

,run_id,timestep,time_s,pressure_A,pressure_B,pressure_C,velocity_A,velocity_B,velocity_C,label,scenario
0,0,0,0.0,0.007812,0.007812,0.007812,2.951487,2.963145,2.949855,0,normal
1,0,1,0.1,117246.900000,58469.720000,359.566900,2.947447,2.952487,2.940701,0,normal
2,0,2,0.2,101416.800000,50775.800000,300.451400,2.948791,2.953200,2.941500,0,normal
3,0,3,0.3,88645.910000,44387.200000,253.575200,2.949434,2.953305,2.941705,0,normal
4,0,4,0.4,82063.770000,41106.620000,224.899000,2.949710,2.953260,2.941774,0,normal
5,0,5,0.5,78725.090000,39421.350000,208.390800,2.949816,2.953166,2.941785,0,normal
6,0,6,0.6,77244.900000,38672.990000,198.614300,2.949866,2.953143,2.941715,0,normal
7,0,7,0.7,76636.210000,38373.760000,193.834900,2.949900,2.953127,2.941692,0,normal
8,0,8,0.8,76362.770000,38233.630000,191.487400,2.949922,2.953126,2.941698,0,normal
9,0,9,0.9,76258.660000,38173.730000,189.966700,2.949936,2.953139,2.941686,0,normal


### SYNTHETIC DATA GENERATOR

In [46]:
# CFD-derived constants
SS_MEAN = {
    "pressure_A": 76520.142450,
    "pressure_B": 38311.024783,
    "pressure_C":   189.487823,
    "velocity_A":     2.949948,
    "velocity_B":     2.953167,
    "velocity_C":     2.941687,
}

SS_STD = {
    "pressure_A": 1.25455962,
    "pressure_B": 0.80119845,
    "pressure_C": 0.00881103,
    "velocity_A": 0.00000045,
    "velocity_B": 0.00000055,
    "velocity_C": 0.00000089,
}

# Transient peaks at t=0.1s (from CFD)
PEAK = {
    "pressure_A": 117246.90,
    "pressure_B":  58469.72,
    "pressure_C":    359.57,
    "velocity_A":    2.9515,
    "velocity_B":    2.9631,
    "velocity_C":    2.9499,
}

# Noise sigma = 3× CFD std (pressure floor 0.5 Pa, velocity floor 0.001 m/s)
SIGMA = {
    "pressure_A": max(SS_STD["pressure_A"] * 3, 0.5),
    "pressure_B": max(SS_STD["pressure_B"] * 3, 0.5),
    "pressure_C": max(SS_STD["pressure_C"] * 3, 0.5),
    "velocity_A": max(SS_STD["velocity_A"] * 3, 0.001),
    "velocity_B": max(SS_STD["velocity_B"] * 3, 0.001),
    "velocity_C": max(SS_STD["velocity_C"] * 3, 0.001),
}

TIMESTEPS  = 701          # steps 0..700
DT         = 0.1          # seconds per step
SS_START_T = 10.0         # transient ends here


def simulate_run(run_id: int, rng: np.random.Generator) -> pd.DataFrame:
    """
    Generate one complete synthetic Normal run (701 rows).
    """
    times = np.round(np.arange(TIMESTEPS) * DT, 1)

    peak_scale  = rng.uniform(0.98, 1.02)     # ±2% peak pressure variation
    decay_scale = rng.uniform(0.95, 1.05)     # ±5% decay rate variation
    # global steady-state pressure offset (pump speed jitter ±0.05%)
    global_scale = rng.normal(1.0, 0.0005)

    records = []
    for step in range(TIMESTEPS):
        t = times[step]

        if t == 0.0:
            pA = 0.0
            pB = 0.0
            pC = 0.0
            vA = SS_MEAN["velocity_A"] * global_scale
            vB = SS_MEAN["velocity_B"] * global_scale
            vC = SS_MEAN["velocity_C"] * global_scale

        elif t <= SS_START_T:
            # decay constant fitted to CFD shape (≈1.5 s time constant)
            tau   = 1.5 * decay_scale
            alpha = np.exp(-t / tau)

            pA = (SS_MEAN["pressure_A"] * global_scale
                  + (PEAK["pressure_A"] * peak_scale - SS_MEAN["pressure_A"]) * alpha
                  + rng.normal(0, SIGMA["pressure_A"] * (1 + alpha)))

            pB = (SS_MEAN["pressure_B"] * global_scale
                  + (PEAK["pressure_B"] * peak_scale - SS_MEAN["pressure_B"]) * alpha
                  + rng.normal(0, SIGMA["pressure_B"] * (1 + alpha)))

            pC = (SS_MEAN["pressure_C"] * global_scale
                  + (PEAK["pressure_C"] * peak_scale - SS_MEAN["pressure_C"]) * alpha
                  + rng.normal(0, SIGMA["pressure_C"] * (1 + alpha)))

            vA = SS_MEAN["velocity_A"] * global_scale + rng.normal(0, SIGMA["velocity_A"])
            vB = SS_MEAN["velocity_B"] * global_scale + rng.normal(0, SIGMA["velocity_B"])
            vC = SS_MEAN["velocity_C"] * global_scale + rng.normal(0, SIGMA["velocity_C"])

        else:
            pA = SS_MEAN["pressure_A"] * global_scale + rng.normal(0, SIGMA["pressure_A"])
            pB = SS_MEAN["pressure_B"] * global_scale + rng.normal(0, SIGMA["pressure_B"])
            pC = SS_MEAN["pressure_C"] * global_scale + rng.normal(0, SIGMA["pressure_C"])
            vA = SS_MEAN["velocity_A"] * global_scale + rng.normal(0, SIGMA["velocity_A"])
            vB = SS_MEAN["velocity_B"] * global_scale + rng.normal(0, SIGMA["velocity_B"])
            vC = SS_MEAN["velocity_C"] * global_scale + rng.normal(0, SIGMA["velocity_C"])

        records.append({
            "run_id":     run_id,
            "timestep":   step,
            "time_s":     t,
            "pressure_A": pA,
            "pressure_B": pB,
            "pressure_C": pC,
            "velocity_A": vA,
            "velocity_B": vB,
            "velocity_C": vC,
            "label":      0,
            "scenario":   "normal",
        })

    return pd.DataFrame(records)

N_SYNTHETIC_RUNS = 74      # change this to get more data
rng = np.random.default_rng(seed=42)

syn_runs = []
for i in range(1, N_SYNTHETIC_RUNS + 1):
    run_df = simulate_run(run_id=i, rng=rng)
    syn_runs.append(run_df)
    print(f"  Run {i} generated: {len(run_df)} rows")

syn_df = pd.concat(syn_runs, ignore_index=True)
print(f"\nSynthetic total : {len(syn_df)} rows  ({N_SYNTHETIC_RUNS} runs × {TIMESTEPS} steps)")

  Run 1 generated: 701 rows
  Run 2 generated: 701 rows
  Run 3 generated: 701 rows
  Run 4 generated: 701 rows
  Run 5 generated: 701 rows
  Run 6 generated: 701 rows
  Run 7 generated: 701 rows
  Run 8 generated: 701 rows
  Run 9 generated: 701 rows
  Run 10 generated: 701 rows
  Run 11 generated: 701 rows
  Run 12 generated: 701 rows
  Run 13 generated: 701 rows
  Run 14 generated: 701 rows
  Run 15 generated: 701 rows
  Run 16 generated: 701 rows
  Run 17 generated: 701 rows
  Run 18 generated: 701 rows
  Run 19 generated: 701 rows
  Run 20 generated: 701 rows
  Run 21 generated: 701 rows
  Run 22 generated: 701 rows
  Run 23 generated: 701 rows
  Run 24 generated: 701 rows
  Run 25 generated: 701 rows
  Run 26 generated: 701 rows
  Run 27 generated: 701 rows
  Run 28 generated: 701 rows
  Run 29 generated: 701 rows
  Run 30 generated: 701 rows
  Run 31 generated: 701 rows
  Run 32 generated: 701 rows
  Run 33 generated: 701 rows
  Run 34 generated: 701 rows
  Run 35 generated: 701

### COMBINING DATA

In [47]:
normal_df = pd.concat([cfd_df, syn_df], ignore_index=True)

n_runs = normal_df["run_id"].nunique()
n_rows = len(normal_df)

print("=" * 55)
print("  NORMAL DATASET — FINAL SUMMARY")
print("=" * 55)
print(f"  Total runs   : {n_runs}  (1 real CFD + {N_SYNTHETIC_RUNS} synthetic)")
print(f"  Rows per run : {TIMESTEPS}  (t=0 -> 70s, dt=0.1s)")
print(f"  Total rows   : {n_rows}")
print(f"  Scenario     : normal")
print(f"  Label        : 0")
print(f"  Columns      : {list(normal_df.columns)}")
print("=" * 55)

out_path = OUTPUT_DIR / "normal_combined.csv"
normal_df.to_csv(out_path, index=False, float_format="%.6f")
print(f"\n  Saved -> {out_path}")

  NORMAL DATASET — FINAL SUMMARY
  Total runs   : 75  (1 real CFD + 74 synthetic)
  Rows per run : 701  (t=0 -> 70s, dt=0.1s)
  Total rows   : 52575
  Scenario     : normal
  Label        : 0
  Columns      : ['run_id', 'timestep', 'time_s', 'pressure_A', 'pressure_B', 'pressure_C', 'velocity_A', 'velocity_B', 'velocity_C', 'label', 'scenario']

  Saved -> /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/raw/normal_combined.csv


In [48]:
normal_df.shape

(52575, 11)